# 2D optimization

Configure an optimization, run it with live progress, and compare the
optimized section against the original. This demo keeps it small
(`maxiter=10`) and cleans up after itself.

In [ ]:
from pytakeoff import TakeoffClient

# Ran `python -m pytakeoff`? Leave this line exactly as it is: the all-x
# placeholder counts as no key, so your saved credentials (or the
# TAKEOFF_API_KEY env var) are used automatically.
# Otherwise, paste your own key here - GUI: Account -> API Keys.
API_KEY = "tk_xxxxxxxx_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

client = TakeoffClient(api_key=API_KEY)
print(f"Connected to {client.base_url} as {client.username}")

project = client.projects.current() or client.projects.open(client.projects.list()[0]["name"])
project

## Configure

One objective (best glide at Re 10⁶, α = 4°) plus a geometric and an
aerodynamic constraint — same fields as the GUI tables.

In [ ]:
import matplotlib.pyplot as plt

section = project.foil_sections()[0]

opt = project.create_optimization_2d(
    "notebook_opt",
    initial_section=section.name,
    solver="NN",
    optimizer_config={"maxiter": 10, "tol": 1e-3},
)
opt.set_objectives([
    {"func": "maxGlide", "Re": 1e6, "alpha": 4.0, "solve_for": "fixed"},
])
opt.set_constraints(
    geo=[{"variable": "get_tc", "operator": ">", "value": 0.10}],
    aero=[{"variable": "Cl", "operator": ">", "value": 0.8,
           "Re": 1e6, "alpha": 8.0, "solve_for": "fixed"}],
)
opt

## Run it

In [ ]:
response = opt.run(on_progress=lambda pct, msg: print(f"{float(pct):5.1f}%  {msg or ''}"),
                   timeout=None)
r = opt.result()
obj = r["opt_data"]["objective_1"]
print(f"\nCl={obj['Cl']:.4f}  Cd={obj['Cd']:.6f}  glide={obj['Cl']/obj['Cd']:.1f}")

## Original vs optimized shape

Save the result into the project to get a real `FoilSection`, then plot both outlines.

In [ ]:
saved = opt.save_result()
optimized = project.foil_section(id=saved["section_id"])

plt.figure(figsize=(9, 3))
plt.plot(*zip(*section.points()), label=f"original ({section.name})")
plt.plot(*zip(*optimized.points()), label=f"optimized ({optimized.name})")
plt.axis("equal"); plt.legend(); plt.grid(alpha=0.3)
plt.title(f"maxGlide at Re=1e6, alpha=4  ->  Cl/Cd = {obj['Cl']/obj['Cd']:.1f}")
plt.show()

## Clean up (demo only — keep them in your own runs!)

In [ ]:
project.delete_entity("FoilSection", id=optimized.id)
project.delete_entity("OptiAeroFoil", id=opt.id)
client.close()
print("deleted", optimized.name, "and", opt.name)